# Incremental CPM Scheduling

This notebook demonstrates the incremental scheduling path added in Wave 3 (issue #8).
When only a few tasks change, `schedule(project, changed_task_ids=[...])` computes
only the affected subgraph instead of re-running the full forward/backward pass.

**Key properties**:
- Results are **identical** to a full recompute
- Fast for small change sets on large projects
- Falls back to full recompute automatically when `changed_task_ids` is `None`
  or when the change set exceeds 25% of tasks

**Install**
```bash
pip install trueppm-scheduler
```

In [ ]:
import time
from datetime import date, timedelta
from trueppm_scheduler import (
    Calendar, Dependency, DependencyType, Project, Task, schedule,
)

## 1. Build a medium-sized FS chain project

500 tasks in a single dependency chain is a worst-case scenario for the
forward pass (no parallelism). We use this to measure timing differences.

In [ ]:
def make_fs_chain(n: int, start: date = date(2026, 1, 5)) -> Project:
    """Build an n-task FS chain — worst-case for forward-pass traversal."""
    tasks = [
        Task(id=f"t{i}", name=f"Task {i}", duration=timedelta(days=1))
        for i in range(n)
    ]
    deps = [
        Dependency(predecessor_id=f"t{i}", successor_id=f"t{i+1}")
        for i in range(n - 1)
    ]
    return Project(
        id="bench",
        name=f"{n}-task chain",
        start_date=start,
        tasks=tasks,
        dependencies=deps,
        calendar=Calendar(),
    )

project = make_fs_chain(500)
print(f"Project: {len(project.tasks)} tasks, {len(project.dependencies)} dependencies")

## 2. Full recompute vs. incremental — timing comparison

The incremental path only traverses the ancestry + descendant set of
the changed tasks. For a 5-task change in a 500-task chain it visits at
most `5 + descendants` — much smaller than 500.

In [ ]:
# Warm up (import-time costs)
schedule(project)

# Full recompute
t0 = time.perf_counter()
result_full = schedule(project)
full_ms = (time.perf_counter() - t0) * 1000

# Incremental: only tasks t100..t104 changed
changed = [f"t{i}" for i in range(100, 105)]
t0 = time.perf_counter()
result_incr = schedule(project, changed_task_ids=changed)
incr_ms = (time.perf_counter() - t0) * 1000

print(f"Full recompute:  {full_ms:.1f} ms")
print(f"Incremental (5 changed tasks): {incr_ms:.1f} ms")
if incr_ms > 0:
    print(f"Speedup: {full_ms / incr_ms:.1f}x")

## 3. Result equivalence

Incremental and full results must agree on every task date and float value.
The CI fuzz test (`tests/test_incremental_equivalence.py`) asserts this
across 1000 random change scenarios.

In [ ]:
mismatches = []
for tf, ti in zip(result_full.tasks, result_incr.tasks):
    if tf.early_start != ti.early_start or tf.early_finish != ti.early_finish:
        mismatches.append(tf.id)

if mismatches:
    print(f"MISMATCH on tasks: {mismatches}")
else:
    print("All task dates match between full and incremental result ✓")

assert result_full.project_finish == result_incr.project_finish
print(f"Project finish: {result_full.project_finish}")

## 4. Automatic full-recompute fallback

Two conditions trigger an automatic fallback to full recompute:

1. `changed_task_ids=None` — no change hint provided (explicit full recompute)
2. `len(changed_task_ids) / len(project.tasks) > 0.25` — more than 25% of tasks
   changed; the overhead of subgraph extraction exceeds the savings


In [ ]:
# Case 1: explicit None → full recompute
result_none = schedule(project, changed_task_ids=None)
assert result_none.project_finish == result_full.project_finish
print("changed_task_ids=None falls back to full recompute ✓")

# Case 2: > 25% changed → full recompute  
many_changed = [f"t{i}" for i in range(0, 200)]  # 200/500 = 40%
result_many = schedule(project, changed_task_ids=many_changed)
assert result_many.project_finish == result_full.project_finish
print("Large change set (40%) falls back to full recompute ✓")

## 5. Using incremental scheduling in practice

The typical call site is inside a Celery task that handles a `PATCH /tasks/<id>/`
API request:

```python
# packages/api/src/trueppm_api/apps/scheduling/tasks.py (simplified)
from trueppm_scheduler import schedule

@app.task
def recalculate_schedule(project_id: str, changed_task_ids: list[str] | None = None):
    project = build_scheduler_project(project_id)  # serialise from DB
    result  = schedule(project, changed_task_ids=changed_task_ids)
    apply_result_to_db(project_id, result)          # write ES/EF back
    broadcast_schedule_updated(project_id)          # WS event
```

The API view passes the changed task IDs when the PATCH touches `duration`,
`early_start`, or dependencies — the three fields that affect the CPM graph.

## 6. Performance target and CI guard

The Wave 3 bench target is:

| Scenario | Target |
|----------|--------|
| 500-task chain, 5-task incremental change | < 200 ms |
| 500-task chain, full recompute | < 2 s |

The `scheduler:bench` CI job in `.gitlab-ci.yml` enforces these limits.
Artifacts from each run are stored for 90 days to detect gradual regressions
(> 20% slower than the previous artifact).

```bash
# Run locally
cd packages/scheduler
pytest tests/test_bench.py -v
```

In [ ]:
# Quick local bench to verify targets
import statistics

RUNS = 10

full_times  = []
incr_times  = []
changed_5   = [f"t{i}" for i in range(50, 55)]

for _ in range(RUNS):
    t0 = time.perf_counter()
    schedule(project)
    full_times.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    schedule(project, changed_task_ids=changed_5)
    incr_times.append((time.perf_counter() - t0) * 1000)

print(f"Full recompute   — median {statistics.median(full_times):.1f} ms, "
      f"p95 {sorted(full_times)[int(RUNS * 0.95)]:.1f} ms  (target: < 2000 ms)")
print(f"Incremental (5t) — median {statistics.median(incr_times):.1f} ms, "
      f"p95 {sorted(incr_times)[int(RUNS * 0.95)]:.1f} ms  (target: < 200 ms)")